In [47]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import joblib

print("All libraries imported successfully!")

All libraries imported successfully!


In [48]:
df = pd.read_csv(r"C:\Users\vaibh\Downloads\Delhi_AQI_Dataset.csv")
print(df.columns)


Index(['City', 'Date', 'AQI', 'PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3',
       'Unnamed: 9', 'Unnamed: 10'],
      dtype='object')


In [50]:
X = df[['PM2.5', 'PM10', 'NO2']].values  # Use the available columns as features
y = df['AQI'].values

In [10]:
X = df[['latitude', 'longitude', 'pollutant_avg']].values  # Features
y = df['pollutant_avg'].values  # Target: pollutant_avg

In [51]:
df = df.dropna()

In [52]:
scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))

# Reshape X for LSTM input: [samples, time steps, features]
X_scaled = X_scaled.reshape((X_scaled.shape[0], 1, X_scaled.shape[1]))

In [53]:
timesteps = 60
X_lstm, y_lstm = [], []

for i in range(timesteps, len(X_scaled)):
    X_lstm.append(X_scaled[i - timesteps:i])  # Last 60 timesteps for each sample
    y_lstm.append(y_scaled[i])  # Predict the value at the next timestep

X_lstm, y_lstm = np.array(X_lstm), np.array(y_lstm)

In [54]:
print(f"Prepared data for LSTM: {X_lstm.shape}, {y_lstm.shape}")

Prepared data for LSTM: (2480, 60, 1, 3), (2480, 1)


In [55]:
model = Sequential()

# Add the LSTM layer
model.add(LSTM(units=50, return_sequences=False, input_shape=(X_lstm.shape[1], X_lstm.shape[2])))

c:\Users\vaibh\Desktop\project\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [56]:
model.add(Dense(units=1))  # Only one output (pollutant_avg)

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Summarize the model architecture
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 50)             │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,451 (40.82 KB)

 Trainable params: 10,451 (40.82 KB)

 Non-trainable params: 0 (0.00 B)

In [62]:
# LSTM model definition example
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(50, activation='relu', input_shape=(1, 3)),
    tf.keras.layers.Dense(1)
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')


In [63]:
# Reshape the data for LSTM input: [samples, time_steps, features]
# Here we are considering 1 time step per sample, with 3 features (PM2.5, PM10, NO2)
X_lstm = X_scaled.reshape((X_scaled.shape[0], 1, num_features))

# Now, train the model with the corrected shape
history = model.fit(X_lstm, y_scaled, epochs=100, batch_size=32, validation_split=0.2)

# Save the model and scaler for later use
model.save('aqi_lstm_model.h5')  # Save the trained model
joblib.dump(scaler_X, 'scaler_X.pkl')  # Save the feature scaler
joblib.dump(scaler_y, 'scaler_y.pkl')  # Save the target scaler


Epoch 1/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1382 - val_loss: 0.0503
Epoch 2/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0297 - val_loss: 0.0080
Epoch 3/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0074 - val_loss: 0.0049
Epoch 4/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0045 - val_loss: 0.0027
Epoch 5/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0024 - val_loss: 0.0014
Epoch 6/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0014 - val_loss: 7.5008e-04
Epoch 7/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 7.3935e-04 - val_loss: 4.9238e-04
Epoch 8/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 5.0577e-04 - val_loss: 4.0508e-04
Epoch 9/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.2726e-04 - val_loss: 3.7803e-04
Epoch 10/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.0772e-04 - val_loss: 3.7030e-04
Epoch 11/100
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.6323e-04 - val_loss: 3.5937e-04
Epoch 12/100
64/64 

['scaler_y.pkl']

In [64]:
import tensorflow as tf


In [65]:
import tensorflow as tf
import joblib

# Load the saved model and scaler (if needed later)
model = tf.keras.models.load_model('aqi_lstm_model.h5')
scaler_X = joblib.load('scaler_X.pkl')
scaler_y = joblib.load('scaler_y.pkl')

In [66]:
latitude = 28.7041   # Example latitude (Delhi)
longitude = 77.1025  # Example longitude (Delhi)
pollutant_avg = 120

In [67]:
new_data = np.array([[latitude, longitude, pollutant_avg]])

# Scale the new data using the previously saved scaler
scaled_data = scaler_X.transform(new_data)

In [68]:
scaled_data = scaled_data.reshape((1, 1, scaled_data.shape[1]))  # 1 timestep, number of features

# Make the prediction
predicted_aqi = model.predict(scaled_data)

# Inverse transform the prediction to get the original AQI value
predicted_aqi = scaler_y.inverse_transform(predicted_aqi)

print("Predicted AQI:", predicted_aqi)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
Predicted AQI: [[95.66549]]


In [69]:
print(df.columns)


Index(['City', 'Date', 'AQI', 'PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3',
       'Unnamed: 9', 'Unnamed: 10'],
      dtype='object')


In [71]:
# Use 'AQI' as the target variable
y = df['AQI'].values  # Target variable (AQI)

# Use other columns as features, for example, PM2.5, PM10, NO2
X = df[['PM2.5', 'PM10', 'NO2']].values  # Features (PM2.5, PM10, NO2)

print(df.head())


Empty DataFrame
Columns: [City, Date, AQI, PM2.5, PM10, NO2, SO2, CO, O3, Unnamed: 9, Unnamed: 10]
Index: []


In [73]:
# Check for missing values in the selected features
print(df[['PM2.5', 'PM10', 'NO2']].isnull().sum())

# Drop rows with missing values in the features
df_clean = df.dropna(subset=['PM2.5', 'PM10', 'NO2'])


PM2.5    0
PM10     0
NO2      0
dtype: int64


In [77]:
# Example values for PM2.5, PM10, and NO2
PM2_5_value = 12.5  # Replace with actual value
PM10_value = 25.3   # Replace with actual value
NO2_value = 32.1    # Replace with actual value

# Now, create the new data input as a numpy array
new_data = np.array([[PM2_5_value, PM10_value, NO2_value]])  # Example new data input

# Transform the new data using the fitted scaler
X_input_scaled = scaler_X.transform(new_data)  # Scale the new input data

# Reshape the new data if necessary (for LSTM input)
X_input_scaled = X_input_scaled.reshape(X_input_scaled.shape[0], 1, X_input_scaled.shape[1])

# Predict the output using the trained model
predictions = model.predict(X_input_scaled)


NotFittedError: This MinMaxScaler instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [78]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)


In [79]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Initialize the model
model = Sequential()

# Add an LSTM layer
model.add(LSTM(units=50, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])))

# Add the output layer
model.add(Dense(1))

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=32)


Epoch 1/10


c:\Users\vaibh\Desktop\project\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1330  
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0105
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0048
Epoch 4/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0017  
Epoch 5/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 5.7293e-04  
Epoch 6/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 2.1163e-04  
Epoch 7/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.5082e-04
Epoch 8/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.4175e-04
Epoch 9/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.2853e-04
Epoch 10/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.2177e-04  


In [80]:
df = df.dropna(subset=['pollutant_avg'])


KeyError: ['pollutant_avg']

In [81]:
df['pollutant_avg'].fillna(df['pollutant_avg'].mean(), inplace=True)  # Fill NaN with mean value


KeyError: 'pollutant_avg'

In [82]:
df['pollutant_avg'] = df['pollutant_avg'].interpolate()


KeyError: 'pollutant_avg'

In [83]:
# Save the trained model
model.save('aqi_lstm_model.h5')

# Save the scalers for later use (prediction)
import joblib
joblib.dump(scaler_X, 'scaler_X.pkl')
joblib.dump(scaler_y, 'scaler_y.pkl')


['scaler_y.pkl']

In [84]:
# Evaluate the model on test data
loss = model.evaluate(X_test, y_test)
print("Test Loss:", loss)


16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.0366e-04  
Test Loss: 0.00010525719699217007


In [85]:
# Make predictions on the test data
y_pred = model.predict(X_test)

# Reverse the scaling of predictions and actual values
y_pred_rescaled = scaler_y.inverse_transform(y_pred)
y_test_rescaled = scaler_y.inverse_transform(y_test)

# Print some predicted vs actual values
for true, pred in zip(y_test_rescaled[:10], y_pred_rescaled[:10]):
    print(f"True: {true}, Predicted: {pred}")


16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
True: [335.], Predicted: [334.76297]
True: [164.], Predicted: [160.95322]
True: [103.], Predicted: [105.702866]
True: [313.], Predicted: [311.07776]
True: [281.], Predicted: [277.2484]
True: [84.], Predicted: [89.33504]
True: [358.], Predicted: [359.86612]
True: [222.], Predicted: [217.03987]
True: [187.], Predicted: [182.80067]
True: [160.], Predicted: [157.20862]


In [86]:
# Assuming your data is already preprocessed into X_train and y_train

model = Sequential()
model.add(LSTM(units=50, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dense(1))  # Output layer

model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X_train, y_train, epochs=10, batch_size=32)

# Save the trained model
model.save('aqi_lstm_model.h5')

# Save the scalers
joblib.dump(scaler_X, 'scaler_X.pkl')
joblib.dump(scaler_y, 'scaler_y.pkl')


Epoch 1/10


c:\Users\vaibh\Desktop\project\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1523  
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0105
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0048  
Epoch 4/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0019
Epoch 5/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 6.5965e-04
Epoch 6/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 2.5479e-04
Epoch 7/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.8203e-04
Epoch 8/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.6585e-04
Epoch 9/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.5146e-04
Epoch 10/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.3819e-04


['scaler_y.pkl']

In [87]:
model.save('aqi_lstm_model.h5')